## Check the Noise Floor
This notebook identifies the noise floor for our sample data since the data will naturally fluctuate due to normal variance or measurement errors. This offers a baseline so we can avoid false positive drifts and hopefully separate a real drift signal from the noise floor. 

**Method:**
We'll score two random halves of the same-issue week using the Unsupervised method's scoring steps. We use the same week because the same week = no change, so whatever change score we get from that we'll consider as our noise floor. 

**Output:** 
noise_floor.csv: one row per issue-week with its noise floor. It's NaN when the week has less than 100 tweets.

In [1]:
import numpy as np
import pandas as pd
import time
import warnings
import sys
from sklearn.exceptions import ConvergenceWarning

sys.path.append("../SandboxNotebooks")

from shift_scoring import (
    K, ISSUE_KEYWORDS, tag_issues, filter_issue_tweets, make_issue_vectorizer,
    fit_frames, shift_between, run_issue,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
df = pd.read_parquet("../sampleddata.parquet")
df = tag_issues(df)
features = pd.read_csv("../DataFiles/tk_issue_week_features.csv")
print(df.shape)

(465146, 26)


In [3]:
# Check A: my issue tweet counts should match Tanisha's table exactly
my_counts = df[[f"issue_{i}" for i in ISSUE_KEYWORDS]].sum().rename(lambda c: c.removeprefix("issue_"))
her_counts = features.groupby("issue")["n_tweets_issue"].sum()
print(pd.DataFrame({"mine": my_counts, "hers": her_counts}))

                 mine   hers
abortion         3192   3192
democracy        8319   8319
economy         16934  16934
foreign_policy  18668  18668
immigration     16178  16178


In [4]:
# Check B: the shared file should reproduce Tanisha's shift scores for every issue
for issue in ISSUE_KEYWORDS:
    mine = run_issue(df, issue)
    hers = features[features["issue"] == issue][["week", "shift_score"]].dropna()
    compare = mine.merge(hers, left_on="period", right_on="week", suffixes=("_mine", "_hers"))
    gap = (compare["shift_score_mine"] - compare["shift_score_hers"]).abs().max()
    print(f"{issue}: {len(compare)} weeks compared, biggest difference = {gap}")

immigration: 30 weeks compared, biggest difference = 0.0
economy: 30 weeks compared, biggest difference = 0.0
abortion: 29 weeks compared, biggest difference = 0.0
democracy: 30 weeks compared, biggest difference = 0.0
foreign_policy: 30 weeks compared, biggest difference = 0.0


##### Make sure shift_scoring.py reproduces Unsupervised shift scores

In [5]:
mine = run_issue(df, "immigration")
hers = features[features["issue"] == "immigration"][["week", "shift_score"]]
compare = mine.merge(hers, left_on="period", right_on="week", suffixes=("_mine", "_hers"))
print((compare["shift_score_mine"] - compare["shift_score_hers"]).abs().max())

0.0


##### Make sure my keyword matching is same as Unsupervised feature table

In [6]:
# If my keyword guess matches her changes. I can delete once she pushes her notebook.

mine = df[[f"issue_{i}" for i in ISSUE_KEYWORDS]].sum().rename(lambda c: c.removeprefix("issue_"))
hers = features.groupby("issue")["n_tweets_issue"].sum()

# Building the DataFrame first lines the rows up by issue name,
# so the comparison works even though the two lists were in different orders
check = pd.DataFrame({"mine": mine, "hers": hers})
check["match"] = check["mine"] == check["hers"]
print(check)

                 mine   hers  match
abortion         3192   3192   True
democracy        8319   8319   True
economy         16934  16934   True
foreign_policy  18668  18668   True
immigration     16178  16178   True


In [7]:
def noise_floor(issue, week, k=K, n_repeats=20, seed=42):
    """Score two random halves of the same issue-week with Tanisha's scoring steps.
    Same week = no real change, so the score is chance alone (the noise floor)."""
    rng = np.random.default_rng(seed)

    # Same tweet filter and vocab as run_issue (shared in shift_scoring.py)
    tw = filter_issue_tweets(df, issue)
    vec = make_issue_vectorizer().fit(tw["cleanedTweet"])
    X_week = vec.transform(tw.loc[tw["week"] == week, "cleanedTweet"])

    # Each half needs 50+ tweets, so the week needs 100+ (same per-group rule as run_issue)
    n = X_week.shape[0]
    if n < 2 * k * 10:
        return np.nan

    # I'll average over many random splits for a more stable result
    scores = []
    for _ in range(n_repeats):
        idx = rng.permutation(n)
        A = fit_frames(X_week[idx[: n // 2]], k)
        B = fit_frames(X_week[idx[n // 2 :]], k)
        scores.append(shift_between(A, B)["shift_score"])
    return np.mean(scores)

##### Test on one week

In [8]:
# Test on week 12
start = time.time()
test_noise = noise_floor("immigration", 12, n_repeats=3)
elapsed = time.time() - start

# The week-to-week shift score for the same issue-week
real_score = features.loc[
    (features["issue"] == "immigration") & (features["week"] == 12), "shift_score"
].iloc[0]

print(f"Noise floor (same week, two halves): {test_noise:.3f}")
print(f"Shift score (week 11 to 12):    {real_score:.3f}")

Noise floor (same week, two halves): 0.168
Shift score (week 11 to 12):    0.363


In [ ]:
# Use the feature table as the list of issue-weeks to score,
# so the noise floor lines up with the Unsupervised shift scores

rows = []
for issue, week in features[["issue", "week"]].itertuples(index=False):
    rows.append({
        "issue": issue,
        "week": week,
        "noise_floor": noise_floor(issue, week),   # NaN if too few tweets
    })

noise_df = pd.DataFrame(rows)
noise_df.to_csv("../DataFiles/noise_floor.csv", index=False)  # index=False skips the row-number column
noise_df.head()

,issue,week,noise_floor
0,abortion,0,0.42000
1,abortion,1,0.40225
2,abortion,2,NaN
3,abortion,3,NaN
4,abortion,4,NaN


**Summary:** For each issue, how many weeks have a noise floor and how often is the shift score higher than the noise floor. 

In [10]:
# Join noise floors onto the real shift scores, row for row
check_nf = features[["issue", "week", "n_tweets_issue", "shift_score"]].merge(
    noise_df, on=["issue", "week"]
)

# Real change above the noise floor (negative = real score is within chance)
check_nf["shift_minus_noise"] = check_nf["shift_score"] - check_nf["noise_floor"]

summary = check_nf.groupby("issue").agg(
    weeks=("week", "size"),
    weeks_with_noise_floor=("noise_floor", lambda s: s.notna().sum()),
    avg_tweets=("n_tweets_issue", "mean"),
    avg_shift=("shift_score", "mean"),
    avg_noise=("noise_floor", "mean"),
    # share of scored weeks where the real change beats chance
    pct_above_noise=("shift_minus_noise", lambda s: (s.dropna() > 0).mean()),
).round(3)
print(summary)

                weeks  weeks_with_noise_floor  avg_tweets  avg_shift  \
issue                                                                  
abortion           31                      15     102.968      0.387   
democracy          31                      31     268.355      0.318   
economy            31                      31     546.258      0.274   
foreign_policy     31                      31     602.194      0.302   
immigration        31                      31     521.871      0.277   

                avg_noise  pct_above_noise  
issue                                       
abortion            0.371            0.571  
democracy           0.334            0.433  
economy             0.288            0.367  
foreign_policy      0.271            0.567  
immigration         0.289            0.400  


### What the results say

For every issue, the typical week-to-week shift score is about the same size as the noise floor. Averages are nearly the same. Economy averages 0.274 for real shifts and 0.288 for noise. The other issues look similar. Only foreign policy’s average real shift (0.302) is clearly above its average noise (0.271).
Real scores are greater than chance about half the time. 

Abortion is the weakest issue. Only 15 of 31 weeks had enough tweets (100 or more) to measure a noise floor, and its noise is the highest of any issue (0.371). Its high shift score is mostly small-sample noise.

### What this means for the project

Real narrative shifts should be rare. In most weeks nothing new happens, so we’d expect most weeks to be near the noise floor and a few to stand clearly above it. These averages can’t tell us yet whether those few clear weeks exist so we'll check that next.

### What the results tell us:

The shift_score can’t be used on its own as a drift signal. We'll use shift_minus_noise as a feature. Next, for the Supervised portion, we'll label based on reading the tweets instead of using these scores. The Supervised model’s goal is to find out whether the noisy scores have a useful signal when we verify them against the labels. Abortion should be dropped or marked low-confidence. 

### Why it matters

Without this check, we may report any fluctations as narrative shifts and get false positives. In general the change scores were mostly at chance level, so we corrected for noise before modeling.

### Are there any clearly above-chance weeks?

In [ ]:
# Count weeks where the real shift beats chance by 0.05 or more
margin = 0.05
check_nf["clearly_above"] = check_nf["shift_minus_noise"] >= margin

print(check_nf.groupby("issue")["clearly_above"].sum())

# List those weeks so we can check them against real-world events
print(check_nf[check_nf["clearly_above"]][["issue", "week", "shift_score", "noise_floor"]]
      .sort_values("shift_score", ascending=False))

issue
abortion           3
democracy          5
economy            5
foreign_policy    11
immigration        6
Name: clearly_above, dtype: int64
              issue  week  shift_score  noise_floor
101  foreign_policy     8        0.442      0.32975
9          abortion     9        0.424      0.36470
26         abortion    26        0.409      0.33255
74          economy    12        0.403      0.33640
145     immigration    21        0.402      0.17130
105  foreign_policy    12        0.395      0.20360
35        democracy     4        0.392      0.24535
106  foreign_policy    13        0.376      0.30810
87          economy    25        0.373      0.27170
136     immigration    12        0.363      0.18485
8          abortion     8        0.357      0.30600
154     immigration    30        0.353      0.29980
130     immigration     6        0.350      0.26475
43        democracy    12        0.342      0.24085
53        democracy    22        0.339      0.27465
81          economy    

### These results will be used to see if they line up with real events.